### Feature Engineering

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNetCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
data_path = Path("../data/processed")

daily_df = pd.read_csv(data_path / "daily_df.csv", parse_dates=["Date"])
interval_df = pd.read_csv(data_path / "interval_df.csv")
staffing_df = pd.read_csv(data_path / "staffing_df.csv")

# need to rebuild the datetime for interval data
interval_df["Month"] = interval_df["Month"].astype(str).str.strip()
interval_df["Day"] = pd.to_numeric(interval_df["Day"], errors="coerce")
interval_df["Interval"] = interval_df["Interval"].astype(str).str.strip()

interval_df["month_num"] = pd.to_datetime(interval_df["Month"], format="%B", errors="coerce").dt.month
interval_df["datetime"] = pd.to_datetime(
    "2024-" + interval_df["month_num"].astype("Int64").astype(str) + "-"
    + interval_df["Day"].astype("Int64").astype(str) + " " + interval_df["Interval"],
    errors="coerce"
)

print("daily:", daily_df.shape)
print("interval:", interval_df.shape)
print("staffing:", staffing_df.shape)

In [ ]:
TARGET = "Call Volume"
N_SPLITS = 5
MIN_VOTE_COUNT = 2  # need at least 2/3 methods to agree
TOP_LGBM_GAIN = 20
TOP_PERM = 20
ELASTIC_THRESHOLD = 1e-6

OUTPUT_DIR = Path("outputs/client_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# columns we dont want as features
DROP_COLS = [TARGET, "Date", "Date_raw", "source_sheet",
             "hour", "minute", "interval_num", "interval_x_dow"]

# might be leakage - commenting out for now but might add back
# LEAKAGE = ["Service Level", "Abandon Rate", "CCT", "volume_x_service", "volume_x_abandon"]

In [ ]:
def get_client_Xy(df_client):
    """prep X and y for one client"""
    df_client = df_client.copy().sort_values("Date").reset_index(drop=True)
    
    drop = [c for c in DROP_COLS + ["client"] if c in df_client.columns]
    X = df_client.drop(columns=drop)
    y = df_client[TARGET]
    
    # drop rows where target is missing
    ok = y.notna()
    X, y = X.loc[ok].copy(), y.loc[ok].copy()
    
    # encode any string cols
    for col in X.columns:
        if X[col].dtype == "object":
            X[col] = X[col].astype("category").cat.codes
    
    # drop useless columns
    X = X.drop(columns=[c for c in X.columns if X[c].isna().all() or X[c].nunique(dropna=True) <= 1], errors='ignore')
    return X, y


def run_lgbm_importance(X, y):
    """get feature importance from lightgbm across folds"""
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    all_imp = []
    for fold, (tr, te) in enumerate(tscv.split(X)):
        m = lgb.LGBMRegressor(objective="regression", n_estimators=800,
            learning_rate=0.03, num_leaves=31, subsample=0.8,
            colsample_bytree=0.8, random_state=42)
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[te], y.iloc[te])], eval_metric="l2")
        all_imp.append(pd.DataFrame({"feature": X.columns, "imp": m.feature_importances_, "fold": fold}))
    
    summary = pd.concat(all_imp).groupby("feature")["imp"].mean().sort_values(ascending=False).reset_index()
    return summary, summary.head(TOP_LGBM_GAIN)["feature"].tolist()


def run_perm_importance(X, y):
    """permutation importance"""
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    all_imp = []
    for fold, (tr, te) in enumerate(tscv.split(X)):
        m = lgb.LGBMRegressor(objective="regression", n_estimators=800,
            learning_rate=0.03, num_leaves=31, subsample=0.8,
            colsample_bytree=0.8, random_state=42)
        m.fit(X.iloc[tr], y.iloc[tr])
        perm = permutation_importance(m, X.iloc[te], y.iloc[te], n_repeats=10,
            random_state=42, scoring="neg_mean_absolute_error")
        all_imp.append(pd.DataFrame({"feature": X.columns, "imp": perm.importances_mean, "fold": fold}))
    
    summary = pd.concat(all_imp).groupby("feature")["imp"].mean().sort_values(ascending=False).reset_index()
    summary = summary[summary["imp"] > 0]
    return summary, summary.head(TOP_PERM)["feature"].tolist()


def run_elasticnet(X, y):
    """elasticnet to find features with nonzero coefficients"""
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
            alphas=np.logspace(-4, 1, 40), cv=tscv, max_iter=20000, random_state=42))
    ])
    pipe.fit(X, y)
    coefs = pipe.named_steps["model"].coef_
    coef_df = pd.DataFrame({"feature": X.columns, "coef": coefs, "abs_coef": np.abs(coefs)})
    coef_df = coef_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)
    selected = coef_df[coef_df["abs_coef"] > ELASTIC_THRESHOLD]["feature"].tolist()
    return coef_df, selected

In [ ]:
client_models = {}
client_selected_features = {}
client_metrics = {}
summary_rows = []

for client in sorted(daily_df["client"].dropna().unique()):
    print(f"\n{'='*50}")
    print(f"CLIENT {client}")
    
    client_dir = OUTPUT_DIR / f"client_{client}"
    client_dir.mkdir(parents=True, exist_ok=True)
    
    df_c = daily_df[daily_df["client"] == client].copy()
    X, y = get_client_Xy(df_c)
    print(f"  rows={len(X)}, features={X.shape[1]}")
    
    if len(X) <= N_SPLITS + 1:
        print(f"  skipping - not enough rows")
        continue
    
    # run all 3 feature selection methods
    gain_df, gain_feats = run_lgbm_importance(X, y)
    perm_df, perm_feats = run_perm_importance(X, y)
    enet_df, enet_feats = run_elasticnet(X, y)
    
    # consensus vote - feature needs 2+ methods to agree
    all_feats = sorted(set(gain_feats) | set(perm_feats) | set(enet_feats))
    votes = pd.DataFrame({"feature": all_feats})
    votes["lgbm"] = votes["feature"].isin(gain_feats).astype(int)
    votes["perm"] = votes["feature"].isin(perm_feats).astype(int)
    votes["enet"] = votes["feature"].isin(enet_feats).astype(int)
    votes["total"] = votes["lgbm"] + votes["perm"] + votes["enet"]
    votes = votes.sort_values(["total", "feature"], ascending=[False, True]).reset_index(drop=True)
    
    selected = votes[votes["total"] >= MIN_VOTE_COUNT]["feature"].tolist()
    if not selected:
        print(f"  no consensus features - skipping")
        continue
    print(f"  selected {len(selected)} features: {selected}")
    
    # evaluate with CV
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    fold_metrics = []
    for fold, (tr, te) in enumerate(tscv.split(X[selected])):
        m = lgb.LGBMRegressor(objective="regression", n_estimators=1000,
            learning_rate=0.03, num_leaves=31, subsample=0.8,
            colsample_bytree=0.8, random_state=42)
        m.fit(X[selected].iloc[tr], y.iloc[tr],
              eval_set=[(X[selected].iloc[te], y.iloc[te])], eval_metric="l2")
        p = m.predict(X[selected].iloc[te])
        fold_metrics.append({
            "fold": fold, 
            "mae": mean_absolute_error(y.iloc[te], p),
            "rmse": np.sqrt(mean_squared_error(y.iloc[te], p)),
            "r2": r2_score(y.iloc[te], p)
        })
    
    metrics_df = pd.DataFrame(fold_metrics)
    print(f"  CV results:")
    print(metrics_df)
    
    # train final model on everything
    final_model = lgb.LGBMRegressor(objective="regression", n_estimators=1000,
        learning_rate=0.03, num_leaves=31, subsample=0.8,
        colsample_bytree=0.8, random_state=42)
    final_model.fit(X[selected], y)
    
    client_models[client] = final_model
    client_selected_features[client] = selected
    client_metrics[client] = metrics_df
    
    # save stuff
    gain_df.to_csv(client_dir / "lgbm_gain.csv", index=False)
    perm_df.to_csv(client_dir / "perm_importance.csv", index=False)
    enet_df.to_csv(client_dir / "elasticnet_coefs.csv", index=False)
    votes.to_csv(client_dir / "feature_votes.csv", index=False)
    metrics_df.to_csv(client_dir / "cv_metrics.csv", index=False)
    
    summary_rows.append({
        "client": client,
        "n_features": len(selected),
        "avg_mae": metrics_df["mae"].mean(),
        "avg_rmse": metrics_df["rmse"].mean(),
        "avg_r2": metrics_df["r2"].mean()
    })

In [ ]:
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "summary.csv", index=False)
print(summary_df)
print("\nmodels trained:", list(client_models.keys()))

In [43]:
print(client_selected_features["A"])
print(client_selected_features["B"])
print(client_selected_features["C"])
print(client_selected_features["D"])

['Abandon Rate', 'CCT', 'Service Level', 'pct_change_7', 'volume_x_abandon', 'volume_x_service', 'Call Volume_lag_1', 'Call Volume_lag_14', 'Call Volume_lag_2', 'Call Volume_lag_7', 'Call Volume_rollstd_3', 'Call Volume_rollstd_7', 'day_of_month', 'day_of_week', 'diff_lag7', 'volatility_ratio']
['Abandon Rate', 'Call Volume_lag_1', 'Call Volume_lag_14', 'Call Volume_lag_21', 'Call Volume_lag_7', 'Service Level', 'diff_lag7', 'volume_x_abandon', 'volume_x_service', 'CCT', 'Call Volume_lag_2', 'Call Volume_lag_28', 'Call Volume_rollstd_7', 'day_of_month', 'day_of_week', 'is_weekend', 'pct_change_7']
['Abandon Rate', 'CCT', 'Call Volume_lag_7', 'Service Level', 'diff_lag7', 'volume_x_abandon', 'volume_x_service', 'Call Volume_lag_1', 'Call Volume_lag_14', 'Call Volume_lag_2', 'Call Volume_lag_28', 'Call Volume_rollmax_3', 'Call Volume_rollmean_3', 'Call Volume_rollmean_30', 'Call Volume_rollmin_3', 'Call Volume_rollmin_7', 'day_of_month', 'is_weekend', 'pct_change_7']
['Call Volume_lag_14